# Задание 11. Версионирование модели кластеризации текста

## Задания
1. Настроить MLflow и DAGsHub
2. Загрузить и очистить текстовый датасет
3. Получить эмбеддинги и уменьшить размерность
4. Подобрать число кластеров по метрикам
5. Провести финальную кластеризацию и оценку качества
6. Подготовить артефакты для версионирования модели

## Импорты и конфигурация

In [ ]:
import os
import re
from contextlib import nullcontext
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import umap.umap_ as umap
from IPython.display import display
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import KMeans
from sklearn.datasets import fetch_20newsgroups
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    adjusted_rand_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    f1_score,
    normalized_mutual_info_score,
    silhouette_score,
)
from sklearn.preprocessing import normalize

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

RANDOM_STATE = 42
BASE_DIR = Path(".")
ARTIFACT_DIR = BASE_DIR / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)

DATASET_CATEGORIES = [
    "comp.graphics",
    "comp.os.ms-windows.misc",
    "comp.sys.ibm.pc.hardware",
    "comp.sys.mac.hardware",
    "rec.autos",
    "rec.motorcycles",
    "sci.med",
    "sci.space",
]
DATASET_REMOVE = ("headers", "footers", "quotes")
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
EMBEDDING_BATCH_SIZE = 64
TFIDF_FALLBACK_COMPONENTS = 256
TOP_TERMS_PER_CLUSTER = 10

UMAP_CLUSTER_PARAMS = {
    "n_components": 10,
    "n_neighbors": 25,
    "min_dist": 0.05,
    "metric": "cosine",
    "random_state": RANDOM_STATE,
}

UMAP_VISUAL_PARAMS = {
    "n_components": 2,
    "n_neighbors": 25,
    "min_dist": 0.1,
    "metric": "cosine",
    "random_state": RANDOM_STATE,
}

KMEANS_BASE_PARAMS = {
    "random_state": RANDOM_STATE,
    "n_init": 20,
    "max_iter": 300,
}
K_CANDIDATES = list(range(max(2, len(DATASET_CATEGORIES) - 3), len(DATASET_CATEGORIES) + 4))

ENABLE_MLFLOW_LOGGING = True
PROMPT_FOR_DAGSHUB_CREDENTIALS = False
MLFLOW_EXPERIMENT_NAME = "module_11_text_clustering"
MLFLOW_RUN_NAME = "umap_kmeans_newsgroups"
REGISTER_MODEL = False
REGISTERED_MODEL_NAME = "module-11-kmeans-clustering"

DAGSHUB_USERNAME = os.getenv("MLFLOW_TRACKING_USERNAME", "")
DAGSHUB_TOKEN = os.getenv("MLFLOW_TRACKING_PASSWORD", "")
DAGSHUB_REPOSITORY = os.getenv("MLFLOW_TRACKING_PROJECTNAME", "")

print("Количество тем:", len(DATASET_CATEGORIES))
print("Темы:", DATASET_CATEGORIES)
print("Кандидаты для числа кластеров:", K_CANDIDATES)
print("MLflow logging requested:", ENABLE_MLFLOW_LOGGING)
print("Artifacts directory:", ARTIFACT_DIR.resolve())

## Вспомогательные функции

In [ ]:
def clean_text(text: str) -> str:
    """Очистить текст от переводов строк, URL, email и лишнего шума."""
    text = text.replace("\n", " ").replace("\t", " ")
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"[\w.+-]+@[\w-]+\.[\w.-]+", " ", text)
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip().lower()


def truncate_text(text: str, limit: int = 180) -> str:
    """Обрезать длинный текст для компактного вывода в таблицах."""
    return text if len(text) <= limit else text[:limit] + "..."


def load_newsgroups_dataset(categories: list[str]) -> pd.DataFrame:
    """Загрузить подмножество 20 Newsgroups и привести его к DataFrame."""
    dataset = fetch_20newsgroups(
        subset="train",
        categories=categories,
        remove=DATASET_REMOVE,
    )

    return pd.DataFrame(
        {
            "raw_text": dataset.data,
            "target": dataset.target,
            "target_name": [dataset.target_names[idx] for idx in dataset.target],
        }
    )


def configure_mlflow_tracking(
    enable_mlflow: bool,
    username: str,
    token: str,
    repository: str,
    experiment_name: str,
    prompt_for_credentials: bool = False,
) -> dict[str, Any]:
    """Настроить MLflow tracking на DAGsHub и вернуть состояние трекинга."""
    state: dict[str, Any] = {
        "enabled": False,
        "reason": "",
        "tracking_uri": "",
        "experiment_name": experiment_name,
        "mlflow": None,
        "register_model": REGISTER_MODEL,
        "registered_model_name": REGISTERED_MODEL_NAME,
    }

    if not enable_mlflow:
        state["reason"] = "MLflow отключён флагом ENABLE_MLFLOW_LOGGING."
        return state

    try:
        import mlflow
    except ImportError:
        state["reason"] = "Пакет mlflow не установлен в текущем окружении."
        return state

    if prompt_for_credentials and (not username or not token or not repository):
        from getpass import getpass

        if not username:
            username = input("Enter your DAGsHub username: ").strip()
        if not token:
            token = getpass("Enter your DAGsHub access token: ").strip()
        if not repository:
            repository = input("Enter your DAGsHub project name: ").strip()

    if not username or not token or not repository:
        state["reason"] = "Не заданы DAGsHub credentials/project name, поэтому логирование пропущено."
        return state

    os.environ["MLFLOW_TRACKING_USERNAME"] = username
    os.environ["MLFLOW_TRACKING_PASSWORD"] = token
    os.environ["MLFLOW_TRACKING_PROJECTNAME"] = repository

    tracking_uri = f"https://dagshub.com/{username}/{repository}.mlflow"
    mlflow.set_tracking_uri(tracking_uri)
    mlflow.set_experiment(experiment_name)
    mlflow.sklearn.autolog(log_models=True, silent=True)

    state.update(
        {
            "enabled": True,
            "reason": "MLflow configured for DAGsHub.",
            "tracking_uri": tracking_uri,
            "mlflow": mlflow,
        }
    )
    return state


def build_text_embeddings(
    texts: pd.Series,
    model_name: str = EMBEDDING_MODEL_NAME,
    batch_size: int = EMBEDDING_BATCH_SIZE,
) -> tuple[np.ndarray, dict[str, Any]]:
    """Получить dense-эмбеддинги: сначала sentence-transformers, иначе fallback."""
    text_list = texts.fillna("").astype(str).tolist()

    try:
        from sentence_transformers import SentenceTransformer  # type: ignore
    except ImportError:
        SentenceTransformer = None

    if SentenceTransformer is not None:
        try:
            model = SentenceTransformer(model_name, device="cpu", local_files_only=True)
            embeddings = model.encode(
                text_list,
                batch_size=batch_size,
                show_progress_bar=True,
                normalize_embeddings=True,
            )
            return np.asarray(embeddings, dtype=np.float32), {
                "backend": "sentence-transformer",
                "label": model_name,
                "note": "Используется локально доступная модель sentence-transformers.",
            }
        except Exception:
            pass

    vectorizer = TfidfVectorizer(
        max_features=20_000,
        min_df=3,
        max_df=0.85,
        stop_words="english",
    )
    tfidf_matrix = vectorizer.fit_transform(text_list)
    n_components = min(
        TFIDF_FALLBACK_COMPONENTS,
        tfidf_matrix.shape[0] - 1,
        tfidf_matrix.shape[1] - 1,
    )
    svd = TruncatedSVD(n_components=n_components, random_state=RANDOM_STATE)
    embeddings = svd.fit_transform(tfidf_matrix)
    embeddings = normalize(embeddings).astype(np.float32)
    return embeddings, {
        "backend": "local-fallback",
        "label": "TF-IDF + TruncatedSVD (offline fallback)",
        "note": "sentence-transformers недоступен, поэтому используется локальный dense fallback.",
    }


def fit_umap_projection(embeddings: np.ndarray, reducer_params: dict[str, Any]) -> tuple[Any, np.ndarray]:
    """Обучить UMAP и получить reduced-представление."""
    reducer = umap.UMAP(**reducer_params)
    reduced = reducer.fit_transform(embeddings)
    return reducer, reduced


def purity_score(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Вычислить purity по кластерным меткам."""
    total = 0
    for cluster in np.unique(y_pred):
        mask = y_pred == cluster
        _, counts = np.unique(y_true[mask], return_counts=True)
        total += counts.max()
    return total / len(y_true)


def align_cluster_labels(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    """Сопоставить cluster ids истинным классам через Hungarian matching."""
    true_labels = np.unique(y_true)
    pred_labels = np.unique(y_pred)
    contingency = np.zeros((len(true_labels), len(pred_labels)), dtype=int)

    for row_idx, true_label in enumerate(true_labels):
        for col_idx, pred_label in enumerate(pred_labels):
            contingency[row_idx, col_idx] = np.sum((y_true == true_label) & (y_pred == pred_label))

    row_ind, col_ind = linear_sum_assignment(contingency.max() - contingency)
    mapping = {pred_labels[col]: true_labels[row] for row, col in zip(row_ind, col_ind)}
    return np.array([mapping.get(label, -1) for label in y_pred])


def evaluate_clustering(y_true: np.ndarray, y_pred: np.ndarray, features: np.ndarray) -> dict[str, float]:
    """Посчитать внутренние и внешние метрики кластеризации."""
    aligned_pred = align_cluster_labels(y_true, y_pred)
    return {
        "silhouette": float(silhouette_score(features, y_pred)),
        "calinski_harabasz": float(calinski_harabasz_score(features, y_pred)),
        "davies_bouldin": float(davies_bouldin_score(features, y_pred)),
        "ari": float(adjusted_rand_score(y_true, y_pred)),
        "nmi": float(normalized_mutual_info_score(y_true, y_pred)),
        "purity": float(purity_score(y_true, y_pred)),
        "accuracy": float(accuracy_score(y_true, aligned_pred)),
        "f1_macro": float(f1_score(y_true, aligned_pred, average="macro")),
    }

def search_cluster_count(
    features: np.ndarray,
    y_true: np.ndarray,
    cluster_counts: list[int],
) -> pd.DataFrame:
    """Подобрать число кластеров и вернуть таблицу метрик по каждому k."""
    rows: list[dict[str, float]] = []
    for cluster_count in cluster_counts:
        model = KMeans(n_clusters=cluster_count, **KMEANS_BASE_PARAMS)
        labels = model.fit_predict(features)
        metrics = evaluate_clustering(y_true, labels, features)
        rows.append(
            {
                "cluster_count": cluster_count,
                "silhouette": metrics["silhouette"],
                "calinski_harabasz": metrics["calinski_harabasz"],
                "davies_bouldin": metrics["davies_bouldin"],
                "ari": metrics["ari"],
                "nmi": metrics["nmi"],
                "purity": metrics["purity"],
                "accuracy": metrics["accuracy"],
                "f1_macro": metrics["f1_macro"],
            }
        )
    return pd.DataFrame(rows)


def build_cluster_summary(df: pd.DataFrame, cluster_labels: np.ndarray) -> pd.DataFrame:
    """Собрать краткий профиль кластера: размер, доминирующая тема и чистота."""
    summary_df = df[["target_name"]].copy()
    summary_df["cluster"] = cluster_labels

    rows: list[dict[str, Any]] = []
    for cluster_id, cluster_part in summary_df.groupby("cluster"):
        dominant_distribution = cluster_part["target_name"].value_counts(normalize=True)
        rows.append(
            {
                "cluster": int(cluster_id),
                "size": int(len(cluster_part)),
                "dominant_topic": dominant_distribution.index[0],
                "dominant_share": float(dominant_distribution.iloc[0]),
                "topic_count": int(cluster_part["target_name"].nunique()),
            }
        )

    return pd.DataFrame(rows).sort_values(["cluster"]).reset_index(drop=True)


def extract_top_terms_by_cluster(
    texts: pd.Series,
    cluster_labels: np.ndarray,
    top_n: int = TOP_TERMS_PER_CLUSTER,
) -> pd.DataFrame:
    """Извлечь наиболее характерные TF-IDF-термы для каждого кластера."""
    vectorizer = TfidfVectorizer(max_features=10_000, min_df=5, stop_words="english")
    matrix = vectorizer.fit_transform(texts)
    feature_names = np.array(vectorizer.get_feature_names_out())

    rows: list[dict[str, Any]] = []
    for cluster_id in sorted(np.unique(cluster_labels)):
        centroid = matrix[cluster_labels == cluster_id].mean(axis=0).A1
        top_idx = centroid.argsort()[-top_n:][::-1]
        rows.append(
            {
                "cluster": int(cluster_id),
                "top_terms": ", ".join(feature_names[top_idx]),
            }
        )

    return pd.DataFrame(rows)


def make_cluster_search_figure(search_df: pd.DataFrame) -> Any:
    """Построить график подбора числа кластеров."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    axes[0].plot(search_df["cluster_count"], search_df["silhouette"], marker="o", color="#4C78A8")
    axes[0].set_title("Silhouette score по k")
    axes[0].set_xlabel("Число кластеров")
    axes[0].set_ylabel("Silhouette")

    axes[1].plot(search_df["cluster_count"], search_df["ari"], marker="o", label="ARI", color="#F58518")
    axes[1].plot(search_df["cluster_count"], search_df["nmi"], marker="o", label="NMI", color="#54A24B")
    axes[1].plot(search_df["cluster_count"], search_df["purity"], marker="o", label="Purity", color="#E45756")
    axes[1].set_title("Внешние метрики по k")
    axes[1].set_xlabel("Число кластеров")
    axes[1].set_ylabel("Score")
    axes[1].legend()

    plt.tight_layout()
    return fig


def make_projection_figure(projection_2d: np.ndarray, labels: np.ndarray, title: str) -> Any:
    """Построить 2D-проекцию объектов с раскраской по меткам."""
    fig, ax = plt.subplots(figsize=(8, 6))
    scatter = ax.scatter(
        projection_2d[:, 0],
        projection_2d[:, 1],
        c=labels,
        s=10,
        cmap="tab10",
        alpha=0.8,
    )
    ax.set_title(title)
    ax.set_xlabel("UMAP-1")
    ax.set_ylabel("UMAP-2")
    legend = ax.legend(*scatter.legend_elements(), title="Label", loc="best")
    ax.add_artist(legend)
    plt.tight_layout()
    return fig


def save_experiment_artifacts(
    search_df: pd.DataFrame,
    metrics: dict[str, float],
    cluster_summary: pd.DataFrame,
    top_terms: pd.DataFrame,
    search_figure: Any,
    predicted_figure: Any,
    true_figure: Any,
) -> list[Path]:
    """Сохранить ключевые артефакты эксперимента на диск."""
    artifact_paths = [
        ARTIFACT_DIR / "cluster_search_metrics.csv",
        ARTIFACT_DIR / "final_metrics.csv",
        ARTIFACT_DIR / "cluster_summary.csv",
        ARTIFACT_DIR / "cluster_top_terms.csv",
        ARTIFACT_DIR / "cluster_search_plot.png",
        ARTIFACT_DIR / "predicted_clusters_projection.png",
        ARTIFACT_DIR / "true_topics_projection.png",
    ]

    search_df.to_csv(artifact_paths[0], index=False)
    pd.DataFrame([metrics]).to_csv(artifact_paths[1], index=False)
    cluster_summary.to_csv(artifact_paths[2], index=False)
    top_terms.to_csv(artifact_paths[3], index=False)
    search_figure.savefig(artifact_paths[4], dpi=150, bbox_inches="tight")
    predicted_figure.savefig(artifact_paths[5], dpi=150, bbox_inches="tight")
    true_figure.savefig(artifact_paths[6], dpi=150, bbox_inches="tight")

    plt.close(search_figure)
    plt.close(predicted_figure)
    plt.close(true_figure)
    return artifact_paths


def run_clustering_experiment(
    df: pd.DataFrame,
    tracking_state: dict[str, Any],
) -> dict[str, Any]:
    """Запустить весь пайплайн эмбеддинги -> UMAP -> KMeans -> метрики -> артефакты."""
    mlflow = tracking_state.get("mlflow")
    run_context = mlflow.start_run(run_name=MLFLOW_RUN_NAME) if tracking_state["enabled"] else nullcontext()

    with run_context:
        embeddings, embedding_info = build_text_embeddings(df["clean_text"])
        cluster_reducer, reduced_embeddings = fit_umap_projection(embeddings, UMAP_CLUSTER_PARAMS)
        search_df = search_cluster_count(reduced_embeddings, df["target"].to_numpy(), K_CANDIDATES)
        best_row = search_df.sort_values(["silhouette", "ari"], ascending=[False, False]).iloc[0]
        best_k = int(best_row["cluster_count"])

        final_model = KMeans(n_clusters=best_k, **KMEANS_BASE_PARAMS)
        predicted_labels = final_model.fit_predict(reduced_embeddings)
        final_metrics = evaluate_clustering(df["target"].to_numpy(), predicted_labels, reduced_embeddings)
        final_metrics["best_k"] = float(best_k)

        _, projection_2d = fit_umap_projection(embeddings, UMAP_VISUAL_PARAMS)
        cluster_summary = build_cluster_summary(df, predicted_labels)
        top_terms = extract_top_terms_by_cluster(df["clean_text"], predicted_labels)

        search_figure = make_cluster_search_figure(search_df)
        predicted_figure = make_projection_figure(
            projection_2d,
            predicted_labels,
            "UMAP-проекция: предсказанные кластеры",
        )
        true_figure = make_projection_figure(
            projection_2d,
            df["target"].to_numpy(),
            "UMAP-проекция: реальные темы",
        )
        artifact_paths = save_experiment_artifacts(
            search_df,
            final_metrics,
            cluster_summary,
            top_terms,
            search_figure,
            predicted_figure,
            true_figure,
        )

        registered_model_uri = ""
        if tracking_state["enabled"]:
            mlflow.log_params(
                {
                    "dataset_name": "20newsgroups",
                    "categories": ", ".join(DATASET_CATEGORIES),
                    "category_count": len(DATASET_CATEGORIES),
                    "embedding_model_requested": EMBEDDING_MODEL_NAME,
                    "embedding_backend": embedding_info["backend"],
                    "umap_cluster_params": str(UMAP_CLUSTER_PARAMS),
                    "umap_visual_params": str(UMAP_VISUAL_PARAMS),
                    "k_candidates": str(K_CANDIDATES),
                    "best_k": best_k,
                }
            )
            mlflow.log_metrics({k: float(v) for k, v in final_metrics.items() if k != "best_k"})
            for artifact_path in artifact_paths:
                mlflow.log_artifact(str(artifact_path))
            mlflow.sklearn.log_model(final_model, artifact_path="kmeans_model")

            if tracking_state.get("register_model"):
                try:
                    registration = mlflow.register_model(
                        model_uri=f"runs:/{mlflow.active_run().info.run_id}/kmeans_model",
                        name=tracking_state["registered_model_name"],
                    )
                    registered_model_uri = registration.name
                except Exception:
                    registered_model_uri = "Регистрация модели пропущена: registry недоступен в текущем запуске."

            run_id = mlflow.active_run().info.run_id
        else:
            run_id = ""

    return {
        "embedding_info": embedding_info,
        "embeddings_shape": embeddings.shape,
        "reduced_embeddings": reduced_embeddings,
        "projection_2d": projection_2d,
        "search_df": search_df,
        "best_k": best_k,
        "predicted_labels": predicted_labels,
        "metrics": final_metrics,
        "cluster_summary": cluster_summary,
        "top_terms": top_terms,
        "artifact_paths": [str(path) for path in artifact_paths],
        "tracking_state": tracking_state,
        "run_id": run_id,
        "registered_model_uri": registered_model_uri,
    }


def build_quality_comment(results: dict[str, Any]) -> str:
    """Сформировать краткий человеческий комментарий по качеству кластеризации."""
    best_k = results["best_k"]
    true_k = len(DATASET_CATEGORIES)
    metrics = results["metrics"]

    if best_k == true_k:
        k_comment = "Алгоритм выбрал ровно столько же кластеров, сколько в датасете реальных тем."
    elif abs(best_k - true_k) <= 1:
        k_comment = "Лучшее число кластеров почти совпало с количеством реальных тем, что выглядит вполне ожидаемо."
    else:
        k_comment = "Лучшее число кластеров заметно отличается от числа реальных тем, значит часть тематик смешивается или дробится."

    if metrics["purity"] >= 0.6:
        purity_comment = "Кластеры достаточно чистые по доминирующей теме."
    elif metrics["purity"] >= 0.45:
        purity_comment = "Кластеры умеренно чистые: хорошо отделяются самые явные темы, а близкие техничные темы смешиваются."
    else:
        purity_comment = "Кластеры заметно смешаны, значит пространства признаков пока недостаточно для чёткого разделения всех тем."

    return (
        f"Лучший k = {best_k} при реальном числе тем {true_k}. {k_comment} "
        f"Silhouette = {metrics['silhouette']:.3f}, ARI = {metrics['ari']:.3f}, NMI = {metrics['nmi']:.3f}, Purity = {metrics['purity']:.3f}. "
        f"{purity_comment}"
    )

# Задание 1. Настройка MLflow и DAGsHub

1. Подготовить параметры подключения
2. Инициализировать MLflow tracking
3. Проверить, будет ли текущий запуск логироваться в DAGsHub

## Подготовьте параметры подключения

In [ ]:
tracking_config_df = pd.DataFrame(
    {
        "parameter": [
            "ENABLE_MLFLOW_LOGGING",
            "PROMPT_FOR_DAGSHUB_CREDENTIALS",
            "MLFLOW_EXPERIMENT_NAME",
            "MLFLOW_RUN_NAME",
            "REGISTER_MODEL",
            "REGISTERED_MODEL_NAME",
            "DAGSHUB_USERNAME provided",
            "DAGSHUB_REPOSITORY provided",
        ],
        "value": [
            ENABLE_MLFLOW_LOGGING,
            PROMPT_FOR_DAGSHUB_CREDENTIALS,
            MLFLOW_EXPERIMENT_NAME,
            MLFLOW_RUN_NAME,
            REGISTER_MODEL,
            REGISTERED_MODEL_NAME,
            bool(DAGSHUB_USERNAME),
            bool(DAGSHUB_REPOSITORY),
        ],
    }
)

display(tracking_config_df)
print("Комментарий:")
print("- В ноутбуке уже есть весь код для DAGsHub + MLflow, но сами креды берутся из env vars или через prompt.")
print("- Если креды не заданы, pipeline всё равно выполнится локально без логирования, что удобно для отладки.")

## Инициализируйте MLflow tracking

In [ ]:
tracking_state = configure_mlflow_tracking(
    enable_mlflow=ENABLE_MLFLOW_LOGGING,
    username=DAGSHUB_USERNAME,
    token=DAGSHUB_TOKEN,
    repository=DAGSHUB_REPOSITORY,
    experiment_name=MLFLOW_EXPERIMENT_NAME,
    prompt_for_credentials=PROMPT_FOR_DAGSHUB_CREDENTIALS,
)

tracking_state_df = pd.DataFrame(
    {
        "field": ["enabled", "reason", "tracking_uri", "experiment_name"],
        "value": [
            tracking_state["enabled"],
            tracking_state["reason"],
            tracking_state["tracking_uri"] or "local-only mode",
            tracking_state["experiment_name"],
        ],
    }
)

display(tracking_state_df)

# Задание 2. Загрузка и очистка данных

1. Загрузить 20 newsgroups по 8 темам
2. Посмотреть баланс классов и примеры текстов
3. Очистить текст от шума

## Загрузите датасет 20 newsgroups

In [ ]:
news_df = load_newsgroups_dataset(DATASET_CATEGORIES)
class_distribution = (
    news_df["target_name"].value_counts().rename_axis("topic").reset_index(name="count")
)

sample_raw = news_df[["target_name", "raw_text"]].head(5).copy()
sample_raw["raw_text"] = sample_raw["raw_text"].apply(lambda text: truncate_text(text, 220))

display(pd.DataFrame({"metric": ["rows", "topics"], "value": [len(news_df), news_df['target_name'].nunique()]}))
display(class_distribution)
display(sample_raw)

print("Комментарий:")
print("- В выборке 8 реальных тематик: часть из них близки между собой (несколько computer-категорий), поэтому задача кластеризации не совсем тривиальна.")
print("- Уже на этом шаге видно, что тексты содержат шум: обрывки переписки, артефакты пунктуации и лишние пробелы.")

## Очистите текст

In [ ]:
clean_df = news_df.copy()
clean_df["clean_text"] = clean_df["raw_text"].apply(clean_text)
clean_df = clean_df.loc[clean_df["clean_text"].ne("")].reset_index(drop=True)
clean_df["raw_length"] = clean_df["raw_text"].str.len()
clean_df["clean_length"] = clean_df["clean_text"].str.len()

clean_summary = pd.DataFrame(
    {
        "metric": [
            "rows after cleaning",
            "mean raw length",
            "mean clean length",
            "median clean length",
        ],
        "value": [
            len(clean_df),
            round(clean_df["raw_length"].mean(), 1),
            round(clean_df["clean_length"].mean(), 1),
            round(clean_df["clean_length"].median(), 1),
        ],
    }
)

examples_df = clean_df[["target_name", "raw_text", "clean_text"]].sample(3, random_state=RANDOM_STATE).copy()
examples_df["raw_text"] = examples_df["raw_text"].apply(lambda text: truncate_text(text, 180))
examples_df["clean_text"] = examples_df["clean_text"].apply(lambda text: truncate_text(text, 180))

display(clean_summary)
display(examples_df)

print("Комментарий:")
print("- Очистка удаляет URL, email, табы, переводы строк и лишнюю пунктуацию; после этого текст становится стабильнее для эмбеддингов и кластеризации.")
print("- Содержательная часть текста при этом сохраняется: видны технические термины и тематические слова по каждой группе.")

# Задание 3. Эмбеддинги и снижение размерности

1. Получить dense-эмбеддинги текста
2. Уменьшить размерность через UMAP
3. Подготовить представление для кластеризации и визуализации

## Запустите полный пайплайн эксперимента

In [ ]:
experiment_results = run_clustering_experiment(clean_df, tracking_state)

experiment_summary_df = pd.DataFrame(
    {
        "field": [
            "embedding backend",
            "embedding label",
            "embeddings shape",
            "best_k",
            "run_id",
        ],
        "value": [
            experiment_results["embedding_info"]["backend"],
            experiment_results["embedding_info"]["label"],
            str(experiment_results["embeddings_shape"]),
            experiment_results["best_k"],
            experiment_results["run_id"] or "no mlflow run",
        ],
    }
)

display(experiment_summary_df)
print(experiment_results["embedding_info"]["note"])

## Краткий комментарий к эмбеддингам

In [ ]:
print("Комментарий:")
print("- По заданию предпочтительна модель sentence-transformers; если она доступна локально, ноутбук использует именно её.")
print("- Для безопасного офлайн-запуска предусмотрен dense fallback через TF-IDF + TruncatedSVD, чтобы пайплайн оставался воспроизводимым без лишних ручных действий.")
print("- UMAP применяется уже к dense-векторам, чтобы получить компактное пространство признаков для KMeans и отдельную 2D-проекцию для визуального анализа.")

# Задание 4. Подбор числа кластеров

1. Просмотреть метрики для разных значений k
2. Выбрать лучший вариант по Silhouette score
3. Сопоставить его с реальным количеством тем

## Сравните метрики для разных k

In [ ]:
search_df = experiment_results["search_df"].copy()
display(search_df)

search_fig = make_cluster_search_figure(search_df)
plt.show()

print("Комментарий:")
print("- Для выбора числа кластеров используем Silhouette score как основную внутреннюю метрику, как и требуется в задании.")
print("- Дополнительно смотрим на ARI, NMI и Purity: это помогает понять, насколько найденные кластеры согласуются с реальными темами." )

## Выберите лучшее число кластеров

In [ ]:
true_topic_count = len(DATASET_CATEGORIES)
best_k = experiment_results["best_k"]

print(f"Реальное число тем в датасете: {true_topic_count}")
print(f"Лучшее число кластеров по Silhouette score: {best_k}")
print("Комментарий:")
if abs(best_k - true_topic_count) <= 1:
    print("- Лучшее k практически совпало с числом реальных тем. Это хороший признак: структура данных действительно поддерживает близкое разбиение.")
else:
    print("- Лучшее k заметно отличается от числа реальных тем. Это значит, что часть тем либо смешивается, либо дробится на несколько подкластеров.")

# Задание 5. Финальная кластеризация и оценка качества

1. Посчитать внутренние метрики
2. Посчитать внешние метрики
3. Визуализировать кластеры и проанализировать их состав

## Внутренние и внешние метрики

In [ ]:
metrics_df = pd.DataFrame([experiment_results["metrics"]]).T.reset_index()
metrics_df.columns = ["metric", "value"]
display(metrics_df)

print("Комментарий:")
print("- Внутренние метрики оценивают компактность и разделимость кластеров без знания истинных меток.")
print("- Внешние метрики сравнивают найденные кластеры с настоящими темами newsgroups и показывают, насколько хорошо unsupervised-разбиение приближает реальную структуру данных.")

## Визуализация кластеров

In [ ]:
predicted_fig = make_projection_figure(
    experiment_results["projection_2d"],
    experiment_results["predicted_labels"],
    "UMAP-проекция: предсказанные кластеры",
)
plt.show()

true_fig = make_projection_figure(
    experiment_results["projection_2d"],
    clean_df["target"].to_numpy(),
    "UMAP-проекция: реальные темы",
)
plt.show()

## Профиль кластеров и характерные термы

In [ ]:
cluster_profile_df = experiment_results["cluster_summary"].copy()
top_terms_df = experiment_results["top_terms"].copy()

display(cluster_profile_df)
display(top_terms_df)

print("Комментарий:")
print("- Доминирующая тема и её доля показывают, насколько кластер монолитен или, наоборот, смешан.")
print("- Топ-термы помогают быстро понять семантику кластера: по ним видно, где модель хорошо поймала предметную область, а где склеила похожие темы.")

## Вывод о качестве кластеризации

In [ ]:
print(build_quality_comment(experiment_results))

most_pure_clusters = cluster_profile_df.sort_values("dominant_share", ascending=False).head(3)
most_mixed_clusters = cluster_profile_df.sort_values("dominant_share", ascending=True).head(3)

print()
print("Самые чистые кластеры:")
display(most_pure_clusters)
print("Самые смешанные кластеры:")
display(most_mixed_clusters)

print("Комментарий:")
print("- Обычно лучше всего отделяются тематики с яркой предметной лексикой, например космос, медицина или мотоциклы.")
print("- Сильнее всего смешиваются близкие computer-категории: у них много общей терминологии, поэтому даже хорошие эмбеддинги не всегда разводят их идеально.")

# Задание 6. MLflow и версионирование модели

1. Проверить, что подготовлены артефакты эксперимента
2. Убедиться, что run и метрики могут быть отправлены в DAGsHub
3. Зафиксировать статус логирования модели

## Что было подготовлено для MLflow

In [ ]:
artifacts_df = pd.DataFrame({"artifact_path": experiment_results["artifact_paths"]})
display(artifacts_df)

mlflow_status_df = pd.DataFrame(
    {
        "field": [
            "tracking enabled",
            "tracking uri",
            "run id",
            "registered model status",
        ],
        "value": [
            experiment_results["tracking_state"]["enabled"],
            experiment_results["tracking_state"]["tracking_uri"] or "local-only mode",
            experiment_results["run_id"] or "run not created",
            experiment_results["registered_model_uri"] or "model registration skipped",
        ],
    }
)

display(mlflow_status_df)

print("Комментарий:")
print("- Артефакты уже сохранены локально: таблица подбора k, итоговые метрики, профили кластеров, термы и графики проекций.")
print("- Если заданы креды DAGsHub и установлен mlflow, тот же запуск логируется в Experiments -> MLflow UI без дополнительных изменений кода.")
print("- Для register_model предусмотрен отдельный флаг REGISTER_MODEL: он отключён по умолчанию, чтобы локальная отладка не падала без registry-поддержки.")

# Итог

1. В ноутбуке реализован полный pipeline для кластеризации текста: загрузка данных, очистка, dense-эмбеддинги, UMAP, подбор числа кластеров, KMeans и оценка качества.
2. Подсчитаны все требуемые внутренние метрики (`Silhouette`, `Calinski-Harabasz`, `Davies-Bouldin`) и набор внешних метрик (`ARI`, `NMI`, `Purity`, `Accuracy`, `F1 macro`).
3. Подготовлен код для MLflow + DAGsHub: tracking URI, run, логирование параметров, метрик, артефактов и модели.
4. Ноутбук остаётся воспроизводимым даже без DAGsHub credentials и без установленных `sentence-transformers`/`mlflow`: для локальной отладки есть безопасные fallback-сценарии.